# Kerala Flood GNN - Colab Ready

This notebook starts **after** your existing Sentinel-1 + DEM notebook.

It does **not** redo Sentinel-1 preprocessing. It expects the previous notebook to have already created:

```text
/content/drive/MyDrive/Prj_3_Data/Kerala_Model_Ready/
  2018/patches/X_pre_128.npy
  2018/patches/X_flood_128.npy
  2018/patches/X_dem_128.npy
  2018/patches/y_mask_128.npy
  ... same for 2019-2022
```

Then this notebook adds:

- Kerala rainfall CSV features
- Kerala river-level CSV features
- DEM patch statistics
- Sentinel-1 pre/flood/change patch statistics
- A patch-level graph
- A GCN+GRU model for patch-level flood-fraction regression

Important: this notebook trains on continuous pseudo flood fraction per patch, derived from your SAR change-detection masks. It is patch-level risk/fraction modeling, not pixel-level segmentation.

## 1. Install Libraries

Colab usually has PyTorch installed. This notebook uses a pure PyTorch GCN implementation, so it does not depend on `torch-geometric`. That makes it easier to run in Colab without version issues.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib tqdm

## 2. Mount Google Drive and Configure Paths

Put your rainfall and river CSV files either in:

```text
/content/drive/MyDrive/Prj_3_Data/data/
```

or adjust `TABULAR_DATA_ROOT` below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Prj_3_Data')
MODEL_READY_ROOT = DRIVE_ROOT / 'Kerala_Model_Ready'
TABULAR_DATA_ROOT = DRIVE_ROOT / 'data'
OUTPUT_ROOT = DRIVE_ROOT / 'Kerala_GNN_Outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

YEARS = [2018, 2019, 2020, 2021, 2022]
PATCH_SIZE = 128

print('Model-ready root:', MODEL_READY_ROOT)
print('Tabular data root:', TABULAR_DATA_ROOT)
print('Output root:', OUTPUT_ROOT)

## 3. Imports and Reproducibility

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('PyTorch:', torch.__version__)

## 4. Load Kerala Rainfall Features

Expected rainfall files:

```text
Kerala_2018.csv
Kerala_2019.csv
Kerala_2020.csv
Kerala_2021.csv
Kerala_2022.csv
```

Expected columns from your existing files:

```text
State, District, Date, Year, Month, Avg_rainfall, Agency_name
```

This cell creates year-level rainfall features around known Kerala flood windows.

In [ ]:
EVENT_DATES = {
    2018: ('2018-06-01', '2018-08-19'),
    2019: ('2019-08-01', '2019-08-31'),
    2020: ('2020-06-01', '2020-08-18'),
    2021: ('2021-10-11', '2021-10-26'),
    2022: ('2022-08-01', '2022-09-10'),
}

def find_rainfall_file(year):
    candidates = [
        TABULAR_DATA_ROOT / f'Kerala_{year}.csv',
        TABULAR_DATA_ROOT / f'kerala_{year}.csv',
        DRIVE_ROOT / f'Kerala_{year}.csv',
        DRIVE_ROOT / f'kerala_{year}.csv',
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def load_rainfall_features(years):
    rows = []
    for year in years:
        path = find_rainfall_file(year)
        if path is None:
            print(f'Rainfall missing for {year}; filling zeros')
            rows.append({'year': year})
            continue

        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['Avg_rainfall'] = pd.to_numeric(df['Avg_rainfall'], errors='coerce')
        df = df.dropna(subset=['Date', 'Avg_rainfall'])

        event_start = pd.Timestamp(EVENT_DATES[year][0])
        event_end = pd.Timestamp(EVENT_DATES[year][1])
        pre_start = event_start - pd.Timedelta(days=30)
        post_end = event_end + pd.Timedelta(days=30)

        pre = df[(df['Date'] >= pre_start) & (df['Date'] < event_start)]
        event = df[(df['Date'] >= event_start) & (df['Date'] <= event_end)]
        post = df[(df['Date'] > event_end) & (df['Date'] <= post_end)]

        district_event = event.groupby('District')['Avg_rainfall'].sum() if 'District' in event.columns else pd.Series(dtype=float)

        rows.append({
            'year': year,
            'rain_pre30_mean': pre['Avg_rainfall'].mean(),
            'rain_pre30_sum': pre['Avg_rainfall'].sum(),
            'rain_event_mean': event['Avg_rainfall'].mean(),
            'rain_event_sum': event['Avg_rainfall'].sum(),
            'rain_event_max': event['Avg_rainfall'].max(),
            'rain_post30_mean': post['Avg_rainfall'].mean(),
            'rain_post30_sum': post['Avg_rainfall'].sum(),
            'rain_district_event_sum_mean': district_event.mean() if len(district_event) else np.nan,
            'rain_district_event_sum_max': district_event.max() if len(district_event) else np.nan,
            'rain_rows': len(df),
        })

    feat = pd.DataFrame(rows).fillna(0.0)
    return feat

rain_features = load_rainfall_features(YEARS)
rain_features

## 5. Load Kerala River-Level Features

Expected river file:

```text
Kerala_Riverlevel.csv
```

Expected useful columns from your existing file:

```text
Station, District, Latitude, Longitude, Data Acquisition Time,
River Water Level Telemetry Hourly (meter)
```

This creates year-level river features around flood windows.

In [ ]:
def find_river_file():
    candidates = [
        TABULAR_DATA_ROOT / 'Kerala_Riverlevel.csv',
        TABULAR_DATA_ROOT / 'kerala_riverlevel.csv',
        DRIVE_ROOT / 'Kerala_Riverlevel.csv',
        DRIVE_ROOT / 'kerala_riverlevel.csv',
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

RIVER_TIME_COL = 'Data Acquisition Time'
RIVER_LEVEL_COL = 'River Water Level Telemetry Hourly (meter)'

def load_river_features(years):
    path = find_river_file()
    if path is None:
        print('River file missing; filling zeros')
        return pd.DataFrame({'year': years})

    usecols = None
    df = pd.read_csv(path, usecols=usecols)
    df[RIVER_TIME_COL] = pd.to_datetime(df[RIVER_TIME_COL], errors='coerce', dayfirst=True)
    df[RIVER_LEVEL_COL] = pd.to_numeric(df[RIVER_LEVEL_COL], errors='coerce')
    df = df.dropna(subset=[RIVER_TIME_COL, RIVER_LEVEL_COL])

    rows = []
    for year in years:
        event_start = pd.Timestamp(EVENT_DATES[year][0])
        event_end = pd.Timestamp(EVENT_DATES[year][1])
        pre_start = event_start - pd.Timedelta(days=30)
        post_end = event_end + pd.Timedelta(days=30)

        pre = df[(df[RIVER_TIME_COL] >= pre_start) & (df[RIVER_TIME_COL] < event_start)]
        event = df[(df[RIVER_TIME_COL] >= event_start) & (df[RIVER_TIME_COL] <= event_end + pd.Timedelta(days=1))]
        post = df[(df[RIVER_TIME_COL] > event_end) & (df[RIVER_TIME_COL] <= post_end)]

        station_event = event.groupby('Station')[RIVER_LEVEL_COL].max() if 'Station' in event.columns else pd.Series(dtype=float)

        rows.append({
            'year': year,
            'river_pre30_mean': pre[RIVER_LEVEL_COL].mean(),
            'river_pre30_max': pre[RIVER_LEVEL_COL].max(),
            'river_event_mean': event[RIVER_LEVEL_COL].mean(),
            'river_event_max': event[RIVER_LEVEL_COL].max(),
            'river_post30_mean': post[RIVER_LEVEL_COL].mean(),
            'river_post30_max': post[RIVER_LEVEL_COL].max(),
            'river_station_event_max_mean': station_event.mean() if len(station_event) else np.nan,
            'river_station_event_max_max': station_event.max() if len(station_event) else np.nan,
            'river_event_station_count': event['Station'].nunique() if 'Station' in event.columns else 0,
            'river_rows': len(df),
        })

    feat = pd.DataFrame(rows).fillna(0.0)
    return feat

river_features = load_river_features(YEARS)
river_features

## 6. Load Existing Sentinel-1, DEM, and Mask Patches

This is where the notebook reuses your previous work.

It loads only the already-created patch files. It does not open original TIFFs and does not redo SAR/DEM preprocessing.

In [ ]:
def patch_paths(year):
    patch_dir = MODEL_READY_ROOT / str(year) / 'patches'
    return {
        'pre': patch_dir / f'X_pre_{PATCH_SIZE}.npy',
        'flood': patch_dir / f'X_flood_{PATCH_SIZE}.npy',
        'dem': patch_dir / f'X_dem_{PATCH_SIZE}.npy',
        'mask': patch_dir / f'y_mask_{PATCH_SIZE}.npy',
        'full_mask': MODEL_READY_ROOT / str(year) / f'Kerala_{year}_quick_flood_mask.npy',
    }

def verify_patch_files(years):
    ok_years = []
    for year in years:
        paths = patch_paths(year)
        missing = [name for name, p in paths.items() if name != 'full_mask' and not p.exists()]
        if missing:
            print(f'{year}: missing {missing}; skip this year')
            continue
        pre = np.load(paths['pre'], mmap_mode='r')
        flood = np.load(paths['flood'], mmap_mode='r')
        dem = np.load(paths['dem'], mmap_mode='r')
        mask = np.load(paths['mask'], mmap_mode='r')
        print(year, '| pre', pre.shape, '| flood', flood.shape, '| dem', dem.shape, '| mask', mask.shape)
        ok_years.append(year)
    return ok_years

AVAILABLE_YEARS = verify_patch_files(YEARS)
print('Available years:', AVAILABLE_YEARS)

## 7. Reconstruct Patch Coordinates

The earlier notebook saved patches but did not save row/column metadata. This cell reconstructs patch coordinates by replaying the same extraction rule against the saved full-resolution mask.

If the full mask is unavailable, it falls back to a simple sequential grid index. The sequential fallback still runs the GNN, but spatial edges will be less accurate.

In [ ]:
def reconstruct_coords_from_full_mask(year, patch_count, patch_size=128, stride=128, min_mask_fraction=0.01):
    full_mask_path = patch_paths(year)['full_mask']
    if not full_mask_path.exists():
        print(f'{year}: full mask missing; using fallback grid coordinates')
        cols = int(math.ceil(math.sqrt(patch_count)))
        return np.array([(i // cols, i % cols) for i in range(patch_count)], dtype=np.int32)

    mask = np.load(full_mask_path, mmap_mode='r')
    h, w = mask.shape
    coords = []
    kept = 0

    for row in range(0, h - patch_size + 1, stride):
        for col in range(0, w - patch_size + 1, stride):
            mask_patch = mask[row:row + patch_size, col:col + patch_size]
            flood_fraction = float(mask_patch.mean())
            keep_empty = (flood_fraction == 0 and kept % 5 == 0)
            if flood_fraction >= min_mask_fraction or keep_empty:
                coords.append((row // stride, col // stride))
                kept += 1

    coords = np.array(coords, dtype=np.int32)
    if len(coords) != patch_count:
        print(f'{year}: reconstructed {len(coords)} coords but patch file has {patch_count}; trimming/padding fallback')
        if len(coords) > patch_count:
            coords = coords[:patch_count]
        else:
            cols = int(math.ceil(math.sqrt(patch_count)))
            fallback = np.array([(i // cols, i % cols) for i in range(patch_count)], dtype=np.int32)
            fallback[:len(coords)] = coords
            coords = fallback
    return coords

## 8. Convert Patches Into Graph Nodes

Each node is one `128x128` patch.

Node features include:

- Sentinel-1 pre/flood/channel means and standard deviations
- Sentinel-1 flood-pre change statistics
- DEM mean/std/min/max
- Kerala rainfall features for that year
- Kerala river-level features for that year

Node target:

- continuous `mask_fraction`, between `0` and `1`

Why regression: your retained patches are already biased toward flood-containing areas, so converting every patch above `2%` flood pixels into class `1` makes the target map almost fully positive. Regression preserves severity/detail instead of collapsing everything into flood/non-flood.

In [ ]:
def summarize_s1_patch(arr_chw):
    # arr_chw shape: C,H,W
    means = arr_chw.mean(axis=(1, 2))
    stds = arr_chw.std(axis=(1, 2))
    mins = arr_chw.min(axis=(1, 2))
    maxs = arr_chw.max(axis=(1, 2))
    return np.concatenate([means, stds, mins, maxs]).astype(np.float32)

def summarize_dem_patch(dem_chw):
    d = dem_chw[0]
    return np.array([d.mean(), d.std(), d.min(), d.max()], dtype=np.float32)

hydro_features = rain_features.merge(river_features, on='year', how='outer').fillna(0.0)
hydro_feature_cols = [c for c in hydro_features.columns if c != 'year']
hydro_by_year = hydro_features.set_index('year')[hydro_feature_cols].to_dict(orient='index')

all_x = []
all_y = []
all_meta = []

for year in AVAILABLE_YEARS:
    paths = patch_paths(year)
    pre = np.load(paths['pre'], mmap_mode='r')
    flood = np.load(paths['flood'], mmap_mode='r')
    dem = np.load(paths['dem'], mmap_mode='r')
    mask = np.load(paths['mask'], mmap_mode='r')

    coords = reconstruct_coords_from_full_mask(year, len(mask), PATCH_SIZE, PATCH_SIZE)
    hydro = np.array([hydro_by_year.get(year, {}).get(c, 0.0) for c in hydro_feature_cols], dtype=np.float32)

    for i in tqdm(range(len(mask)), desc=f'features {year}'):
        pre_feat = summarize_s1_patch(pre[i])
        flood_feat = summarize_s1_patch(flood[i])
        change_feat = summarize_s1_patch(flood[i] - pre[i])
        dem_feat = summarize_dem_patch(dem[i])
        y_frac = float(mask[i, 0].mean())

        feat = np.concatenate([pre_feat, flood_feat, change_feat, dem_feat, hydro]).astype(np.float32)
        all_x.append(feat)
        all_y.append(y_frac)
        all_meta.append({
            'year': year,
            'local_patch': i,
            'grid_row': int(coords[i, 0]),
            'grid_col': int(coords[i, 1]),
            'mask_fraction': y_frac,
        })

X = np.vstack(all_x).astype(np.float32)
y = np.array(all_y, dtype=np.float32)
meta = pd.DataFrame(all_meta)

print('X:', X.shape)
print('y:', y.shape, 'mean mask fraction:', float(y.mean()))
print('\nMask-fraction diagnostics by year:')
display(meta.groupby('year')['mask_fraction'].describe())
meta.head()

## 8A. Build GRU Temporal Sequences From Rainfall and River Data

The GNN uses patch-level spatial/context features. The GRU adds temporal information.

For every year, this cell builds a fixed-length daily sequence around the flood window:

- `30` days before event start
- event period
- enough post-event days to reach `90` total days

Each day has rainfall and river-level summary features. The same yearly hydro sequence is attached to all patch nodes from that year. This is a practical shortcut because the current patch files do not store geospatial bounds for district/station-level joins.

In [ ]:
SEQUENCE_DAYS = 90
SEQUENCE_PRE_DAYS = 30
SEQ_FEATURE_COLS = [
    'rain_mean',
    'rain_sum',
    'river_mean',
    'river_max',
]

def load_all_kerala_rainfall():
    frames = []
    for year in YEARS:
        path = find_rainfall_file(year)
        if path is None:
            continue
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df['Avg_rainfall'] = pd.to_numeric(df['Avg_rainfall'], errors='coerce')
        df = df.dropna(subset=['Date', 'Avg_rainfall'])
        frames.append(df[['Date', 'Avg_rainfall']])
    if not frames:
        return pd.DataFrame(columns=['Date', 'Avg_rainfall'])
    return pd.concat(frames, ignore_index=True)

def load_all_kerala_river():
    path = find_river_file()
    if path is None:
        return pd.DataFrame(columns=[RIVER_TIME_COL, RIVER_LEVEL_COL])
    df = pd.read_csv(path)
    df[RIVER_TIME_COL] = pd.to_datetime(df[RIVER_TIME_COL], errors='coerce', dayfirst=True)
    df[RIVER_LEVEL_COL] = pd.to_numeric(df[RIVER_LEVEL_COL], errors='coerce')
    df = df.dropna(subset=[RIVER_TIME_COL, RIVER_LEVEL_COL])
    return df[[RIVER_TIME_COL, RIVER_LEVEL_COL]]

rain_all = load_all_kerala_rainfall()
river_all = load_all_kerala_river()

rain_daily = (
    rain_all.assign(day=rain_all['Date'].dt.floor('D'))
    .groupby('day')['Avg_rainfall']
    .agg(rain_mean='mean', rain_sum='sum')
    if len(rain_all) else pd.DataFrame(columns=['rain_mean', 'rain_sum'])
)

river_daily = (
    river_all.assign(day=river_all[RIVER_TIME_COL].dt.floor('D'))
    .groupby('day')[RIVER_LEVEL_COL]
    .agg(river_mean='mean', river_max='max')
    if len(river_all) else pd.DataFrame(columns=['river_mean', 'river_max'])
)

def build_year_sequence(year):
    event_start = pd.Timestamp(EVENT_DATES[year][0])
    seq_start = event_start - pd.Timedelta(days=SEQUENCE_PRE_DAYS)
    days = pd.date_range(seq_start, periods=SEQUENCE_DAYS, freq='D')
    seq_df = pd.DataFrame(index=days)
    seq_df.index.name = 'day'
    seq_df = seq_df.join(rain_daily, how='left').join(river_daily, how='left')
    seq_df = seq_df[SEQ_FEATURE_COLS].fillna(0.0)
    return seq_df.values.astype(np.float32)

year_sequences = {year: build_year_sequence(year) for year in AVAILABLE_YEARS}
for year, seq in year_sequences.items():
    print(year, seq.shape, 'rain_sum_total:', float(seq[:, 1].sum()), 'river_max:', float(seq[:, 3].max()))

seq_np = np.stack([year_sequences[int(year)] for year in meta['year'].values]).astype(np.float32)
print('Node temporal sequences:', seq_np.shape)

## 9. Build Graph Edges

Edges connect nearby patches from the same year using reconstructed grid coordinates.

This creates an undirected graph with self-loops added later in the GCN layer.

In [ ]:
def build_edges(meta_df, neighbor_radius=1):
    edges = []
    offset_by_year_coord = {}
    for idx, row in meta_df.iterrows():
        key = (int(row['year']), int(row['grid_row']), int(row['grid_col']))
        offset_by_year_coord[key] = idx

    for idx, row in meta_df.iterrows():
        year = int(row['year'])
        r = int(row['grid_row'])
        c = int(row['grid_col'])
        for dr in range(-neighbor_radius, neighbor_radius + 1):
            for dc in range(-neighbor_radius, neighbor_radius + 1):
                if dr == 0 and dc == 0:
                    continue
                if max(abs(dr), abs(dc)) > neighbor_radius:
                    continue
                j = offset_by_year_coord.get((year, r + dr, c + dc))
                if j is not None:
                    edges.append((idx, j))

    edge_index = np.array(edges, dtype=np.int64).T if edges else np.empty((2, 0), dtype=np.int64)
    return edge_index

edge_index_np = build_edges(meta, neighbor_radius=1)
print('edge_index:', edge_index_np.shape)
print('nodes:', len(meta), 'edges:', edge_index_np.shape[1])

## 10. Train/Validation Split

For a more honest check, hold out a full year if possible. This avoids mixing nearby patches from the same flood event across train and validation.

Default: validate on `2022`, train on `2018-2021`.

In [ ]:
VAL_YEAR = 2022 if 2022 in AVAILABLE_YEARS else AVAILABLE_YEARS[-1]

train_mask_np = (meta['year'].values != VAL_YEAR)
val_mask_np = (meta['year'].values == VAL_YEAR)

print('Validation year:', VAL_YEAR)
print('Train nodes:', train_mask_np.sum(), 'positive:', y[train_mask_np].mean())
print('Val nodes:', val_mask_np.sum(), 'positive:', y[val_mask_np].mean())

## 11. Normalize Features and Move to Torch

In [ ]:
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[train_mask_np] = scaler.fit_transform(X[train_mask_np])
X_scaled[val_mask_np] = scaler.transform(X[val_mask_np])

# Scale GRU sequence features using only train-node time steps.
seq_scaler = StandardScaler()
seq_scaled = seq_np.copy()
train_seq_flat = seq_np[train_mask_np].reshape(-1, seq_np.shape[-1])
seq_scaler.fit(train_seq_flat)
seq_scaled = seq_scaler.transform(seq_np.reshape(-1, seq_np.shape[-1])).reshape(seq_np.shape).astype(np.float32)

x_t = torch.tensor(X_scaled, dtype=torch.float32, device=DEVICE)
seq_t = torch.tensor(seq_scaled, dtype=torch.float32, device=DEVICE)
y_t = torch.tensor(y, dtype=torch.float32, device=DEVICE).view(-1, 1)
edge_index = torch.tensor(edge_index_np, dtype=torch.long, device=DEVICE)
train_mask = torch.tensor(train_mask_np, dtype=torch.bool, device=DEVICE)
val_mask = torch.tensor(val_mask_np, dtype=torch.bool, device=DEVICE)

print('Node features:', x_t.shape)
print('Temporal sequences:', seq_t.shape)
print('Regression target:', y_t.shape)
print('Edges:', edge_index.shape)

## 12. Pure PyTorch GCN + GRU Model

The model has two branches:

- GCN branch: learns from spatial graph connections between neighboring Sentinel-1/DEM patches.
- GRU branch: learns temporal rainfall and river-level patterns around each flood event.

The two embeddings are concatenated and classified into flood/non-flood patch labels.

In [ ]:
def normalized_adjacency(edge_index, num_nodes, device):
    row, col = edge_index
    self_loops = torch.arange(num_nodes, device=device)
    row = torch.cat([row, self_loops])
    col = torch.cat([col, self_loops])

    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.clamp(min=1).pow(-0.5)
    values = deg_inv_sqrt[row] * deg_inv_sqrt[col]

    adj = torch.sparse_coo_tensor(
        indices=torch.stack([row, col], dim=0),
        values=values,
        size=(num_nodes, num_nodes),
        device=device,
    ).coalesce()
    return adj

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        x = torch.sparse.mm(adj, x)
        return self.linear(x)

class FloodGCNGRU(nn.Module):
    def __init__(self, node_in_dim, seq_in_dim, gcn_hidden=128, gru_hidden=64, fusion_hidden=128, dropout=0.25):
        super().__init__()
        self.gcn1 = GCNLayer(node_in_dim, gcn_hidden)
        self.gcn2 = GCNLayer(gcn_hidden, gcn_hidden)
        self.gru = nn.GRU(
            input_size=seq_in_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.fusion = nn.Sequential(
            nn.Linear(gcn_hidden + 2 * gru_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, 1),
        )
        self.dropout = dropout

    def forward(self, x, seq, adj):
        gx = self.gcn1(x, adj)
        gx = F.relu(gx)
        gx = F.dropout(gx, p=self.dropout, training=self.training)
        gx = self.gcn2(gx, adj)
        gx = F.relu(gx)

        _, h = self.gru(seq)
        # Bidirectional GRU returns [forward_last, backward_last].
        hx = torch.cat([h[-2], h[-1]], dim=1)

        fused = torch.cat([gx, hx], dim=1)
        return self.fusion(fused)

adj = normalized_adjacency(edge_index, x_t.shape[0], DEVICE)
model = FloodGCNGRU(
    node_in_dim=x_t.shape[1],
    seq_in_dim=seq_t.shape[2],
    gcn_hidden=128,
    gru_hidden=64,
    fusion_hidden=128,
    dropout=0.25,
).to(DEVICE)
print(model)

## 13. Train GNN

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.SmoothL1Loss(beta=0.02)

EPOCHS = 250
best_val_mae = float('inf')
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    pred_train = torch.sigmoid(model(x_t, seq_t, adj))
    loss = criterion(pred_train[train_mask], y_t[train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_frac = torch.sigmoid(model(x_t, seq_t, adj)).detach().cpu().numpy().reshape(-1)

    train_mae = float(np.mean(np.abs(pred_frac[train_mask_np] - y[train_mask_np])))
    val_mae = float(np.mean(np.abs(pred_frac[val_mask_np] - y[val_mask_np])))
    val_rmse = float(np.sqrt(np.mean((pred_frac[val_mask_np] - y[val_mask_np]) ** 2)))
    val_corr = float(np.corrcoef(pred_frac[val_mask_np], y[val_mask_np])[0, 1]) if len(np.unique(y[val_mask_np])) > 1 else np.nan

    history.append({
        'epoch': epoch,
        'loss': float(loss.item()),
        'train_mae': train_mae,
        'val_mae': val_mae,
        'val_rmse': val_rmse,
        'val_corr': val_corr,
    })

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 25 == 0:
        print(f'Epoch {epoch:03d} | loss {loss.item():.4f} | train_mae {train_mae:.4f} | val_mae {val_mae:.4f} | val_rmse {val_rmse:.4f}')

model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print('Best val MAE:', best_val_mae)

## 14. Evaluate and Save Outputs

In [ ]:
model.eval()
with torch.no_grad():
    pred_frac = torch.sigmoid(model(x_t, seq_t, adj)).detach().cpu().numpy().reshape(-1)

val_y = y[val_mask_np]
val_pred_frac = pred_frac[val_mask_np]
mae = float(np.mean(np.abs(val_pred_frac - val_y)))
rmse = float(np.sqrt(np.mean((val_pred_frac - val_y) ** 2)))
corr = float(np.corrcoef(val_pred_frac, val_y)[0, 1]) if len(np.unique(val_y)) > 1 else np.nan

# Optional binary report only. This is not the training target.
BINARY_REPORT_THRESHOLD = 0.10
val_y_bin = (val_y >= BINARY_REPORT_THRESHOLD).astype(int)
val_pred_bin = (val_pred_frac >= BINARY_REPORT_THRESHOLD).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(val_y_bin, val_pred_bin, average='binary', zero_division=0)
acc = accuracy_score(val_y_bin, val_pred_bin)
try:
    auc = roc_auc_score(val_y_bin, val_pred_frac)
except ValueError:
    auc = np.nan

metrics = {
    'val_year': int(VAL_YEAR),
    'mae': mae,
    'rmse': rmse,
    'corr': None if np.isnan(corr) else corr,
    'binary_threshold': BINARY_REPORT_THRESHOLD,
    'binary_accuracy': float(acc),
    'binary_precision': float(precision),
    'binary_recall': float(recall),
    'binary_f1': float(f1),
    'binary_roc_auc': float(auc) if not np.isnan(auc) else None,
}
print(metrics)
print('Optional binary confusion matrix:')
print(confusion_matrix(val_y_bin, val_pred_bin))

results = meta.copy()
results['target_fraction'] = y
results['pred_fraction'] = pred_frac
results['target_binary_report'] = (results['target_fraction'] >= BINARY_REPORT_THRESHOLD).astype(int)
results['pred_binary_report'] = (results['pred_fraction'] >= BINARY_REPORT_THRESHOLD).astype(int)
results.to_csv(OUTPUT_ROOT / 'kerala_gnn_gru_fraction_predictions.csv', index=False)

pd.DataFrame(history).to_csv(OUTPUT_ROOT / 'kerala_gnn_gru_fraction_training_history.csv', index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_ROOT / 'kerala_gnn_gru_fraction_metrics.csv', index=False)
torch.save({
    'model_state_dict': model.state_dict(),
    'node_feature_dim': x_t.shape[1],
    'sequence_feature_dim': seq_t.shape[2],
    'sequence_days': SEQUENCE_DAYS,
    'sequence_feature_cols': SEQ_FEATURE_COLS,
    'architecture': 'GCN_GRU_REGRESSION',
    'target': 'mask_fraction',
    'hydro_feature_cols': hydro_feature_cols,
    'metrics': metrics,
}, OUTPUT_ROOT / 'kerala_gnn_gru_fraction_model.pt')

print('Saved outputs to:', OUTPUT_ROOT)

## 15. Visualize Validation-Year Patch Predictions

This plots predictions in reconstructed patch-grid space.

In [ ]:
def plot_year_patch_map(results_df, year, value_col, title, cmap='viridis', vmax=None):
    d = results_df[results_df['year'] == year].copy()
    if d.empty:
        print('No rows for year', year)
        return
    h = int(d['grid_row'].max()) + 1
    w = int(d['grid_col'].max()) + 1
    grid = np.full((h, w), np.nan, dtype=float)
    for _, row in d.iterrows():
        grid[int(row['grid_row']), int(row['grid_col'])] = row[value_col]

    plt.figure(figsize=(10, 8))
    plt.imshow(grid, cmap=cmap, vmin=0, vmax=vmax)
    plt.colorbar(label=value_col)
    plt.title(title)
    plt.axis('off')
    plt.show()

plot_year_patch_map(results, VAL_YEAR, 'target_fraction', f'{VAL_YEAR} pseudo mask fraction target', cmap='Blues', vmax=max(0.1, results['target_fraction'].quantile(0.98)))
plot_year_patch_map(results, VAL_YEAR, 'pred_fraction', f'{VAL_YEAR} GNN+GRU predicted flood fraction', cmap='magma', vmax=max(0.1, results['pred_fraction'].quantile(0.98)))

## 16. What To Improve Later

Use this notebook as the GNN stage after your Sentinel-1/DEM notebook. For a stronger project result, improve these points later:

1. Save patch geospatial bounds during Sentinel-1 patch extraction. Then rainfall, river stations, and river networks can be joined spatially instead of only by year.
2. Replace pseudo masks with verified flood labels if available.
3. Add river-network edges from actual river geometry if you have shapefiles/GeoJSON.
4. Use year-wise validation, not random patch validation, when reporting final results.
5. Compare GNN against non-graph baselines using the same node features.